# PlurVA zh/id/si: Soft-label distillation + positional-consistency regularization (from-scratch novelty pipeline)

Colab-ready, self-contained notebook version of `scripts/novel_data.py` +
`scripts/train_novel.py` + `scripts/run_cv.py`. Nothing in this notebook
imports from the local repo's `scripts/` directory -- every function is
inlined below, since a fresh Colab VM won't have it on `sys.path`. Getting
the **data** onto the VM needs no Google Drive setup: the repo is public and
`data/` is only ~14MB, so the setup cell below just `git clone`s it.

**What this implements** (see `NLP Shared Task Findings.md` and the plan at
`~/.claude/plans/can-we-draft-something-frolicking-starlight.md` for full
rationale):

- **Mechanism A -- soft-label KL distillation**: Indonesian's `Gold_Answer`
  is always a 5-annotator vote list (e.g. `"A, A, A, A, C"`), for every row.
  Training targets the full empirical vote distribution instead of a
  collapsed one-hot majority label. zh/si have no vote data, so they
  degenerate to one-hot (a strict generalization of hard-label
  cross-entropy, not a special case).
- **Mechanism B -- positional-invariance consistency regularization**: a
  Jensen-Shannon penalty between an example's predicted A/B/C/D distribution
  and a position-permuted variant's, remapped back to canonical order --
  extends the team's findings doc's proposed-but-unimplemented
  answer-position-permutation augmentation idea into an actual
  training-time regularizer.
- **k-fold cross-validation + final refit**: the dev sets are tiny (zh=790,
  id=366, si=203 rows), so a fixed held-out val split permanently wastes
  data. Instead: k CV rounds (each holding out a different fold) pick the
  iteration budget and give an honest mean +/- std performance estimate,
  then one final run trains on 100% of the data for that many iterations --
  that checkpoint is the actual submission candidate.
- **Continuous metric**: expected probability mass on the correct answer
  (soft-target-weighted), replacing a previous 0/0.5/1 argmax-accuracy
  metric that was too coarse at small batch sizes to be useful.

**Everything you might want to change lives in the single CONFIG cell**
below -- edit that cell, then run all cells in order.

## Setup

**Before running:** Runtime -> Change runtime type -> GPU (T4/A100/L4/etc).

**Getting the data**: no Google Drive needed -- the next cell just
`git clone`s the repo directly (it's public, `data/` is only ~14MB), so a
fresh Colab VM has everything this notebook needs in a few seconds.

**Saving your results**: Colab-hosted VMs are ephemeral -- anything written
to `/content/...` disappears when the session disconnects or times out. A
full CV sweep can run for hours, so the cell after the clone *optionally*
mounts Google Drive just for output (`adapters/`, `results/`) -- skip it if
you'd rather manually download those folders before disconnecting instead.

In [ ]:
%%capture
!pip install -U transformers peft accelerate tqdm safetensors sentencepiece huggingface_hub bitsandbytes

In [ ]:
# %%capture above hides ALL output from the install cell, including any pip
# failure -- so a silent install failure there would otherwise only surface
# later as a confusing ImportError deep inside a training cell, far from its
# actual cause. Check right here instead, where it's obvious what broke.
missing = []
for pkg in ["transformers", "peft", "accelerate", "bitsandbytes", "safetensors"]:
    try:
        __import__(pkg)
    except ImportError as e:
        missing.append(f"{pkg} ({e})")
if missing:
    raise RuntimeError(
        "These packages failed to import after the install cell above -- remove "
        "its leading '%%capture' and re-run it to see the actual pip error:\n  "
        + "\n  ".join(missing)
    )
print("All required packages import cleanly.")

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Schizo00/PlurvaLLM-SharedTask.git"
REPO_ROOT = Path("/content/PlurvaLLM-SharedTask")

if not (REPO_ROOT / "data").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

assert (REPO_ROOT / "data").exists(), (
    f"{REPO_ROOT / 'data'} not found after cloning -- check REPO_URL/REPO_ROOT above "
    "(e.g. if the repo is ever made private, this plain HTTPS clone will fail without credentials)."
)
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
# Optional: persist outputs to Drive across sessions (Colab-hosted VMs lose
# everything under /content when disconnected). Skip this cell -- leave
# OUTPUT_ROOT = REPO_ROOT -- if you'd rather just download results manually.
OUTPUT_ROOT = REPO_ROOT
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/PlurvaLLM-SharedTask")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    print("Saving outputs to Drive:", OUTPUT_ROOT)
except ImportError:
    print("Not running in Colab (or Drive skipped) -- saving outputs locally under", OUTPUT_ROOT)

## Config -- all hyperparameters live here

Edit this one cell for any run. Everything below reads from these globals --
nothing else in the notebook needs touching for a normal experiment.

In [ ]:
# ============================================================
# CONFIG -- everything you might want to change lives here.
# ============================================================

MODEL_ID = "Qwen/Qwen3.5-4B"
SEED = 42
FOLD_SEED = 42                 # controls the CV fold assignment (assign_folds)

# LoRA
LORA_RANK = 16
LORA_ALPHA = 32.0
MAX_SEQ_LENGTH = 768
GRAD_CHECKPOINT = True
MAX_GRAD_NORM = 1.0            # gradient-norm clip before every optimizer step
LOAD_IN_4BIT = True            # QLoRA-style NF4 quantization of the frozen base model. CUDA-only
                                #        (bitsandbytes has no MPS support, so this is a no-op on the Mac
                                #        tier -- pick_device_and_dtype's fp16/bf16 path is used there
                                #        unchanged). Cuts the base model from ~8GB (fp16) to ~2.5-3GB,
                                #        which is what actually matters on a 15GB T4 -- the fp16 weights
                                #        alone leave little headroom for training no matter how small
                                #        the batch is. Set False if you're on a bigger GPU (A100/L4) and
                                #        would rather train against full-precision weights.

# Mechanism toggles (ablation grid -- see NLP Shared Task Findings.md / the approved plan)
HARD_LABELS = False            # True = disable Mechanism A: collapse id's vote distribution to a
                                #        one-hot majority label instead of the full empirical distribution
CONSISTENCY_LAMBDA = 0.5       # Mechanism B weight; 0 disables positional-consistency regularization
                                #        entirely (skips the permuted forward pass for speed)

# CV protocol
CV_FOLDS = 5                   # cloud/CUDA tier default (the Mac-only tier used 3 -- more folds is
                                #        affordable here since a GPU makes each round much faster)
PATIENCE = 5                   # early-stopping patience in evals with no val-loss improvement (0 disables)

# Training loop
PER_LANG_BATCH_SIZE = 2        # KEEP CONSERVATIVE ON A T4 (~15GB VRAM, the free-tier Colab default) --
                                #        with CONSISTENCY_LAMBDA>0 the actual forward batch per language
                                #        is 2x this (original + permuted variant), on top of the base
                                #        4B-param model's ~8GB in fp16, so pushing this past ~2-4 on a
                                #        T4 will OOM. Safe to raise (e.g. 8-16) on an A100/L4 with more
                                #        VRAM headroom -- check `!nvidia-smi` for what you actually have.
ITERS = 300                    # iteration ceiling per CV round (early stopping may end it sooner)
STEPS_PER_REPORT = 10
STEPS_PER_EVAL = 5
VAL_BATCHES = 10
LEARNING_RATE = 1e-4

# Final refit -- after the CV sweep finishes and prints a suggested iteration count, set this and
# re-run the "Final refit" cell at the bottom. Leave None to skip that cell for now.
REFIT_ITERS = None

# Output locations
# OUTPUT_ROOT is REPO_ROOT unless the optional Drive-mount cell above changed it --
# see that cell if you want results to survive a disconnected Colab session.
OUT_BASE_DIR = OUTPUT_ROOT / "results" / "novel_cv_colab"
REFIT_OUT_DIR = OUTPUT_ROOT / "adapters" / "novel_refit"

## Imports

In [ ]:
import json
import os
import random
import re
import shutil
import statistics
import time
from collections import Counter, defaultdict

# Must be set before torch touches the MPS backend (harmless on Colab's CUDA
# runtime, but keeps this notebook portable if run locally on Apple Silicon).
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Reduces CUDA allocator fragmentation (the "reserved but unallocated" memory
# an OOM error reports) -- harmless on MPS/CPU, only affects CUDA's allocator.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LANGS = ["zh", "id", "si"]
LETTERS = "ABCD"

## Ported from `scripts/eval_baseline.py` (prompt templates, gold resolution)

In [ ]:
DATA_DIR = REPO_ROOT / "data"
LANG_FILES = {
    "zh": DATA_DIR / "chinese_dev.jsonl",
    "id": DATA_DIR / "indonesian_dev.jsonl",
    "si": DATA_DIR / "sri_lankan_dev.jsonl",
}

LETTER_RE = re.compile(r"\b([ABCD])\b")
THINK_RE = re.compile(r"^.*?</think>", re.DOTALL)

SI_FIXED_OPTION_C = "\u0db4\u0dd2\u0dc5\u0dd2\u0dad\u0dd4\u0dbb\u0dd4 \u0daf\u0dd9\u0d9a\u0db8 \u0db1\u0dd2\u0dc0\u0dd0\u0dbb\u0daf\u0dd2\u0dba\u0dd2."
SI_FIXED_OPTION_D = "\u0db4\u0dd2\u0dc5\u0dd2\u0dad\u0dd4\u0dbb\u0dd4 \u0daf\u0dd9\u0d9a\u0db8 \u0db1\u0dd2\u0dc0\u0dd0\u0dbb\u0daf\u0dd2 \u0db1\u0ddc\u0dc0\u0dda."

PROMPT_TEMPLATES = {
    "zh": """You are a Simplified Chinese Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Chinese context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.
{scenario_line}Question: {question}
Option A: {option_a}
Option B: {option_b}
Option C: {option_c}
Option D: {option_d}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.""",
    "id": """You are an Indonesian Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Indonesian context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.
{scenario_line}Question: {question}
Option A: {option_a}
Option B: {option_b}
Option C: {option_c}
Option D: {option_d}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.""",
    "si": """You are a Sinhala Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Sri Lankan context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.
{scenario_line}Question: {question}
Option A: {option_a}
Option B: {option_b}
Option C: {option_c}
Option D: {option_d}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.""",
}

SI_GOLD_MAP = {"Both": "C", "0": "D"}


def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_rows(lang):
    return read_jsonl(LANG_FILES[lang])


def resolve_gold_candidates(lang, gold_answer):
    """Gold letter(s) as a list: one item normally, or the tied top letters
    (e.g. ["A", "D"]) when Indonesian's 5-annotator vote has no strict
    majority -- either tied letter counts as correct."""
    gold_answer = gold_answer.strip()
    if lang == "si" and gold_answer in SI_GOLD_MAP:
        return [SI_GOLD_MAP[gold_answer]]
    if "," in gold_answer:
        votes = [v.strip() for v in gold_answer.split(",")]
        counts = Counter(votes)
        top_n = counts.most_common(1)[0][1]
        return sorted(letter for letter, n in counts.items() if n == top_n)
    return [gold_answer]


def resolve_gold(lang, gold_answer):
    """Strict single-answer version (used for eval/scoring, not training):
    None if there's no strict majority."""
    candidates = resolve_gold_candidates(lang, gold_answer)
    return candidates[0] if len(candidates) == 1 else None


def resolve_options(lang, row):
    options = {
        "A": row.get("Option_A", ""),
        "B": row.get("Option_B", ""),
        "C": row.get("Option_C", ""),
        "D": row.get("Option_D", ""),
    }
    if lang == "si":
        options["C"] = SI_FIXED_OPTION_C
        options["D"] = SI_FIXED_OPTION_D
    return options


def build_prompt(lang, row, options):
    scenario = row.get("Scenario", "").strip()
    scenario_line = f"Scenario: {scenario}\n" if scenario else ""
    return PROMPT_TEMPLATES[lang].format(
        scenario_line=scenario_line,
        question=row["Question"],
        option_a=options["A"],
        option_b=options["B"],
        option_c=options["C"],
        option_d=options["D"],
    )


def strip_thinking(text):
    return THINK_RE.sub("", text, count=1).strip()


def extract_letter(text, valid_letters, options=None):
    text = strip_thinking(text)
    for m in LETTER_RE.finditer(text):
        if m.group(1) in valid_letters:
            return m.group(1)
    if options is not None:
        norm = text.strip().lower()
        if norm:
            for letter in valid_letters:
                opt_text = options.get(letter, "").strip().lower()
                if opt_text and (opt_text in norm or norm in opt_text):
                    return letter
    return None

## Ported from `scripts/novel_data.py` (soft-label targets, permutation, CV folds)

In [ ]:
def build_target_dist(lang, gold_answer):
    """Per-example target distribution over {A,B,C,D}, summing to 1.0.

    id: the true empirical 5-annotator vote distribution, for EVERY row (not
    just rows with no strict majority) -- e.g. "A, A, A, A, C" ->
    {A:0.8, B:0.0, C:0.2, D:0.0}.

    zh/si: resolve_gold_candidates degenerates to a single-letter list in
    every observed case, so this returns one-hot -- a strict generalization,
    not a special case: if a genuine zh/si tie ever appeared, mass would
    split evenly across the tied letters instead of being dropped or
    duplicated."""
    gold_answer = gold_answer.strip()
    if lang == "id" and "," in gold_answer:
        votes = [v.strip() for v in gold_answer.split(",")]
        n = len(votes)
        counts = Counter(votes)
        return {l: counts.get(l, 0) / n for l in LETTERS}
    candidates = resolve_gold_candidates(lang, gold_answer)
    p = 1.0 / len(candidates)
    return {l: (p if l in candidates else 0.0) for l in LETTERS}


# zh/id: a small, reviewable pool of permutations of A,B,C,D (identity +
# derangement-ish shuffles) -- findings doc's "3 to 4 permutations per question".
_FIXED_PERMUTATION_POOL = {
    "zh": ["ABCD", "BADC", "CDAB", "DCBA"],
    "id": ["ABCD", "BADC", "CDAB", "DCBA"],
}


def sample_permutation(lang, rng, pool_size=4):
    """old_letter -> new_letter. si only ever swaps A/B -- C/D are the fixed
    "Both correct"/"neither correct" meta-options, not real content."""
    if lang == "si":
        if rng.random() < 0.5:
            return {"A": "A", "B": "B", "C": "C", "D": "D"}
        return {"A": "B", "B": "A", "C": "C", "D": "D"}
    perms = _FIXED_PERMUTATION_POOL[lang][:pool_size]
    chosen = rng.choice(perms)
    return dict(zip(LETTERS, chosen))


def permute_example(options, target_dist, perm):
    new_options = {perm[l]: options[l] for l in LETTERS}
    new_target = defaultdict(float)
    for l, p in target_dist.items():
        new_target[perm[l]] += p
    return new_options, dict(new_target)


def assign_folds(rows, lang, k, seed):
    """Fold index (0..k-1) per row, stratified by majority-vote gold letter
    so each fold keeps roughly balanced label counts. Deterministic."""
    groups = defaultdict(list)
    for i, row in enumerate(rows):
        candidates = resolve_gold_candidates(lang, row["Gold_Answer"])
        groups[candidates[0]].append(i)

    fold_of = [None] * len(rows)
    rng = random.Random(seed)
    for letter in sorted(groups):
        idxs = groups[letter]
        rng.shuffle(idxs)
        for j, i in enumerate(idxs):
            fold_of[i] = j % k
    return fold_of

## Ported from `scripts/train_novel.py` -- tokenization & datasets

In [ ]:
def letter_token_ids(tokenizer):
    ids = {}
    for letter in LETTERS:
        toks = tokenizer(f" {letter}", add_special_tokens=False)["input_ids"]
        if len(toks) != 1:
            raise RuntimeError(
                f"completion ' {letter}' tokenizes to {len(toks)} tokens ({toks}); "
                "the answer-position-finding logic assumes exactly 1 (true for "
                "Qwen/Qwen3.5-4B -- re-verify before swapping base models)."
            )
        ids[letter] = toks[0]
    return ids


def tokenize_example(prompt, completion, tokenizer, max_seq_length):
    messages = [{"role": "user", "content": prompt}]
    try:
        chat_prompt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False, enable_thinking=False,
        )
    except TypeError:
        chat_prompt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False,
        )
    prompt_ids = tokenizer(chat_prompt, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]
    input_ids = prompt_ids + completion_ids
    if len(input_ids) > max_seq_length:
        input_ids = input_ids[-max_seq_length:]
        prompt_len = max(0, len(input_ids) - len(completion_ids))
    else:
        prompt_len = len(prompt_ids)
    labels = [-100] * prompt_len + input_ids[prompt_len:]
    return {"input_ids": input_ids, "labels": labels}


class SoftPromptCompletionDataset:
    """Eagerly tokenizes prompt+completion pairs, plus carries each example's
    target probability distribution over {A,B,C,D}. `completion` is built as
    " {argmax letter}" purely to give the -100 mask a boundary --
    soft_ce_at_answer_position never looks at which token was actually
    appended, only at the mask position it produced (completions are always
    exactly one token, so there's always exactly one such position)."""

    def __init__(self, examples, tokenizer, max_seq_length):
        self.examples = []
        for ex in examples:
            tok_ex = tokenize_example(ex["prompt"], ex["completion"], tokenizer, max_seq_length)
            tok_ex["target_dist"] = [ex["target_dist"][l] for l in LETTERS]
            tok_ex["lang"] = ex["lang"]
            self.examples.append(tok_ex)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def build_lang_examples(lang, rows, hard_labels=False):
    """Per-row dicts carrying everything needed for both the primary
    (unpermuted) tokenized dataset and on-the-fly permuted variants."""
    examples = []
    for row in rows:
        options = resolve_options(lang, row)
        target_dist = build_target_dist(lang, row["Gold_Answer"])
        if hard_labels:
            top_letter = max(target_dist, key=lambda l: (target_dist[l], -LETTERS.index(l)))
            target_dist = {l: (1.0 if l == top_letter else 0.0) for l in LETTERS}
        prompt = build_prompt(lang, row, options)
        letter = max(target_dist, key=target_dist.get)
        examples.append({
            "lang": lang, "row": row, "options": options, "target_dist": target_dist,
            "prompt": prompt, "completion": f" {letter}",
        })
    return examples


def build_permuted_tokenized_example(raw_ex, tokenizer, max_seq_length, rng):
    perm = sample_permutation(raw_ex["lang"], rng)
    new_options, new_target = permute_example(raw_ex["options"], raw_ex["target_dist"], perm)
    prompt = build_prompt(raw_ex["lang"], raw_ex["row"], new_options)
    letter = max(new_target, key=new_target.get)
    tok = tokenize_example(prompt, f" {letter}", tokenizer, max_seq_length)
    tok["target_dist"] = [new_target[l] for l in LETTERS]
    tok["lang"] = raw_ex["lang"]
    return tok, perm


def collate_soft(examples, pad_token_id, device):
    max_len = max(len(e["input_ids"]) for e in examples)
    input_ids = torch.full((len(examples), max_len), pad_token_id, dtype=torch.long)
    labels = torch.full((len(examples), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(examples), max_len), dtype=torch.long)
    target_dist = torch.zeros((len(examples), 4), dtype=torch.float32)
    for i, e in enumerate(examples):
        L = len(e["input_ids"])
        input_ids[i, :L] = torch.tensor(e["input_ids"], dtype=torch.long)
        labels[i, :L] = torch.tensor(e["labels"], dtype=torch.long)
        attention_mask[i, :L] = 1
        target_dist[i] = torch.tensor(e["target_dist"], dtype=torch.float32)
    return {
        "input_ids": input_ids.to(device),
        "attention_mask": attention_mask.to(device),
        "labels": labels.to(device),
        "target_dist": target_dist.to(device),
    }

## Losses & continuous metric

In [ ]:
def soft_ce_at_answer_position(logits_shifted, labels_shifted, target_dist_batch, letter_ids):
    """Soft cross-entropy against a target distribution over {A,B,C,D} at the
    single answer-token position each row's -100 mask exposes. Equivalent to
    KL(target || model) up to a model-independent constant, and numerically
    IDENTICAL to F.cross_entropy when target_dist is one-hot -- a strict
    generalization of hard-label CE, not a special-cased alternative.

    Returns (mean loss, mean continuous metric, logits at the answer
    position) -- the logits are returned so the caller can reuse them for
    consistency_loss without a second forward pass."""
    valid = (labels_shifted != -100)
    pos = valid.float().argmax(dim=1)
    logits_at_pos = logits_shifted[torch.arange(logits_shifted.size(0), device=logits_shifted.device), pos]
    log_probs = F.log_softmax(logits_at_pos, dim=-1)
    letter_log_probs = log_probs[:, letter_ids]  # (B, 4), columns ordered A,B,C,D
    per_example_loss = -(target_dist_batch * letter_log_probs).sum(dim=1)
    with torch.no_grad():
        metric = (target_dist_batch * letter_log_probs.exp()).sum(dim=1)
    return per_example_loss.mean(), metric.mean().item(), logits_at_pos


def perm_to_idx_tensor(perms, device):
    """perms: list of {old_letter: new_letter} dicts, length B. Returns a
    (B,4) long tensor where row b, column i (canonical letter LETTERS[i])
    gives the index within the PERMUTED example's letter ordering where that
    canonical letter's content now sits -- i.e. LETTERS.index(perm[l])."""
    idx = torch.zeros(len(perms), 4, dtype=torch.long)
    for b, perm in enumerate(perms):
        for i, l in enumerate(LETTERS):
            idx[b, i] = LETTERS.index(perm[l])
    return idx.to(device)


def consistency_loss(logits_at_pos_orig, logits_at_pos_perm, perm_idx, letter_ids):
    """Jensen-Shannon divergence (symmetric, bounded -- unlike raw KL, which
    blows up as either side's distribution sharpens toward one-hot late in
    training) between the original example's predicted A/B/C/D distribution
    and the permuted variant's, after remapping the permuted variant's
    distribution back into canonical (original) letter order."""
    q_o = F.softmax(logits_at_pos_orig[:, letter_ids], dim=-1)
    q_p_raw = F.softmax(logits_at_pos_perm[:, letter_ids], dim=-1)
    q_p = torch.gather(q_p_raw, 1, perm_idx)
    m = 0.5 * (q_o + q_p)
    js = (
        0.5 * (q_o * (q_o.clamp_min(1e-8).log() - m.clamp_min(1e-8).log())).sum(1)
        + 0.5 * (q_p * (q_p.clamp_min(1e-8).log() - m.clamp_min(1e-8).log())).sum(1)
    )
    return js.mean()

## Training step (per-language backward, macro-averaged across zh/id/si)

In [ ]:
def lang_step(model, tokenizer, dataset, raw_examples, batch_size, letter_ids, device, rng,
               max_seq_length, pad_token_id, consistency_lambda):
    """One language's contribution to a training step: samples a batch (with
    replacement, so smaller languages still fill a full batch every step),
    builds a permuted variant of each row fresh (a new random permutation
    every time the row is drawn), forwards original+permuted together in ONE
    batched call, and combines soft-CE + JS consistency into a single
    per-language loss."""
    idxs = [rng.randrange(len(dataset)) for _ in range(batch_size)]
    orig_examples = [dataset[i] for i in idxs]

    if consistency_lambda > 0:
        perm_pairs = [build_permuted_tokenized_example(raw_examples[i], tokenizer, max_seq_length, rng)
                      for i in idxs]
        perm_examples = [p[0] for p in perm_pairs]
        perms = [p[1] for p in perm_pairs]
        combined = orig_examples + perm_examples
    else:
        combined = orig_examples

    batch = collate_soft(combined, pad_token_id, device)
    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    logits_shifted = out.logits[:, :-1, :]
    labels_shifted = batch["labels"][:, 1:]
    B = len(orig_examples)

    loss_orig, metric_orig, logits_at_pos_orig = soft_ce_at_answer_position(
        logits_shifted[:B], labels_shifted[:B], batch["target_dist"][:B], letter_ids,
    )

    if consistency_lambda > 0:
        loss_perm, metric_perm, logits_at_pos_perm = soft_ce_at_answer_position(
            logits_shifted[B:], labels_shifted[B:], batch["target_dist"][B:], letter_ids,
        )
        perm_idx = perm_to_idx_tensor(perms, device)
        c_loss = consistency_loss(logits_at_pos_orig, logits_at_pos_perm, perm_idx, letter_ids)
        lang_loss = 0.5 * (loss_orig + loss_perm) + consistency_lambda * c_loss
        metric = 0.5 * (metric_orig + metric_perm)
        c_loss_value = c_loss.item()
    else:
        lang_loss = loss_orig
        metric = metric_orig
        c_loss_value = 0.0

    return lang_loss, metric, c_loss_value


def soft_lang_backward(model, tokenizer, datasets, raw_examples_by_lang, batch_size, letter_ids,
                        device, rng, max_seq_length, pad_token_id, consistency_lambda):
    """Three separate per-language .backward() calls (equal-weight macro
    average, mirroring the shared task's macro-accuracy metric), each freeing
    its forward graph immediately rather than keeping all three alive."""
    total = 0.0
    lang_metric = {}
    lang_closs = {}
    for lang in LANGS:
        lang_loss, metric, c_loss_value = lang_step(
            model, tokenizer, datasets[lang], raw_examples_by_lang[lang], batch_size, letter_ids,
            device, rng, max_seq_length, pad_token_id, consistency_lambda,
        )
        (lang_loss / len(LANGS)).backward()
        total += lang_loss.item()
        lang_metric[lang] = metric
        lang_closs[lang] = c_loss_value
    return total / len(LANGS), lang_metric, lang_closs


@torch.no_grad()
def per_language_soft_validate(model, val_datasets, batch_size, val_batches, letter_ids,
                                pad_token_id, device, rng):
    """Macro (equal-weight) val loss + continuous metric, ORIGINAL variant
    only -- consistency regularization is a training-time inductive bias,
    not part of the quantity used for model selection/early stopping."""
    model.eval()
    lang_losses, lang_metrics = {}, {}
    for lang, ds in val_datasets.items():
        losses, metrics = [], []
        for _ in range(val_batches):
            idxs = [rng.randrange(len(ds)) for _ in range(batch_size)]
            batch = collate_soft([ds[i] for i in idxs], pad_token_id, device)
            out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            logits_shifted = out.logits[:, :-1, :]
            labels_shifted = batch["labels"][:, 1:]
            loss, metric, _ = soft_ce_at_answer_position(
                logits_shifted, labels_shifted, batch["target_dist"], letter_ids,
            )
            losses.append(loss.item())
            metrics.append(metric)
        lang_losses[lang] = sum(losses) / len(losses)
        lang_metrics[lang] = sum(metrics) / len(metrics)
        tqdm.write(f"  val_loss[{lang}]={lang_losses[lang]:.4f} val_metric[{lang}]={lang_metrics[lang]:.4f}")
    model.train()
    macro_loss = sum(lang_losses.values()) / len(lang_losses)
    macro_metric = sum(lang_metrics.values()) / len(lang_metrics)
    return macro_loss, macro_metric, lang_losses, lang_metrics


def save_best_known(model, adapter_path, best_adapter_path):
    """Periodic save mirrors the best-known checkpoint rather than the
    current in-memory weights, so adapter_path is never worse than
    best_adapter_path (matters once early stopping has fired)."""
    if best_adapter_path.exists():
        shutil.copytree(best_adapter_path, adapter_path, dirs_exist_ok=True)
    else:
        model.save_pretrained(str(adapter_path))

## Data split (CV round vs. final refit)

In [ ]:
def load_split(round_idx, cv_folds):
    """round_idx=None -> final refit (100% of dev data, no held-out val).
    Otherwise -> CV round: train = folds != round_idx, val = fold round_idx,
    synchronized across zh/id/si (one shared multilingual model per round)."""
    train_rows, val_rows = {}, {}
    for lang in LANGS:
        rows = load_rows(lang)
        if round_idx is None:
            train_rows[lang] = rows
            val_rows[lang] = []
        else:
            folds = assign_folds(rows, lang, cv_folds, FOLD_SEED)
            train_rows[lang] = [r for r, f in zip(rows, folds) if f != round_idx]
            val_rows[lang] = [r for r, f in zip(rows, folds) if f == round_idx]
    return train_rows, val_rows

## Fresh model + tokenizer loader (one LoRA per round, from the base model -- matches the \"from scratch\" design)

In [ ]:
def pick_device_and_dtype():
    if torch.cuda.is_available():
        device = "cuda"
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    dtype = torch.float16 if device == "cuda" and not torch.cuda.is_bf16_supported() else torch.bfloat16
    return device, dtype


def load_fresh_model_and_tokenizer():
    device, dtype = pick_device_and_dtype()
    quantize = LOAD_IN_4BIT and device == "cuda"  # bitsandbytes is CUDA-only
    print(f"Loading {MODEL_ID} on {device} ({dtype}), 4-bit={quantize} ...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    letter_ids = torch.tensor([letter_token_ids(tokenizer)[l] for l in LETTERS], device=device)

    if quantize:
        # NF4 double-quant: frozen base weights drop from ~8GB (fp16) to
        # ~2.5-3GB, leaving most of a T4's 15GB free for actual training
        # instead of just holding weights. LoRA adapter itself stays in full
        # precision (unaffected -- quantization only touches the frozen base).
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, quantization_config=bnb_config, device_map={"": 0},
        )
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=GRAD_CHECKPOINT)
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype)

    lora_config = LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    if not quantize:
        # prepare_model_for_kbit_training already wires up gradient checkpointing
        # correctly for the quantized path above -- this branch is the plain
        # fp16/bf16 path's equivalent (MPS, CPU, or LOAD_IN_4BIT=False on CUDA).
        if GRAD_CHECKPOINT:
            model.gradient_checkpointing_enable()
            model.enable_input_require_grads()
        model.to(device)
    model.train()
    return model, tokenizer, letter_ids, device

## One round: train (+ validate unless final refit) -- adapted from `train_novel.py`'s main loop

In [ ]:
def round_out_dir(round_idx):
    return REFIT_OUT_DIR if round_idx is None else OUT_BASE_DIR / f"round_{round_idx}"


def run_round(round_idx, iters):
    """round_idx=None -> final refit (100% data, fixed `iters`, no val/early
    stopping). Otherwise -> one CV round (train on folds != round_idx,
    validate on fold round_idx, with early stopping)."""
    is_refit = round_idx is None
    out_dir = round_out_dir(round_idx)
    out_dir.mkdir(parents=True, exist_ok=True)

    train_rows, val_rows = load_split(round_idx, CV_FOLDS)
    raw_examples_by_lang = {
        lang: build_lang_examples(lang, train_rows[lang], hard_labels=HARD_LABELS)
        for lang in LANGS
    }

    model, tokenizer, letter_ids, device = load_fresh_model_and_tokenizer()

    train_datasets = {
        lang: SoftPromptCompletionDataset(raw_examples_by_lang[lang], tokenizer, MAX_SEQ_LENGTH)
        for lang in LANGS
    }
    val_datasets = None
    if not is_refit:
        val_raw = {
            lang: build_lang_examples(lang, val_rows[lang], hard_labels=HARD_LABELS)
            for lang in LANGS
        }
        val_datasets = {
            lang: SoftPromptCompletionDataset(val_raw[lang], tokenizer, MAX_SEQ_LENGTH)
            for lang in LANGS
        }
        for lang in LANGS:
            print(f"{lang}: {len(train_datasets[lang])} train, {len(val_datasets[lang])} val "
                  f"(fold {round_idx}/{CV_FOLDS})")
    else:
        for lang in LANGS:
            print(f"{lang}: {len(train_datasets[lang])} train (100% dev data, final refit)")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    rng = random.Random(SEED)

    adapter_path = out_dir / "adapter"
    best_adapter_path = adapter_path / "best"
    best_val_loss = float("inf")
    best_iter = 0
    stale_evals = 0

    losses, steps, t0 = 0.0, 0, time.time()
    metric_sums = {lang: 0.0 for lang in LANGS}
    last_summary = None  # tracks the most recent eval's summary dict, so a final
                          # "finished": True write at the end doesn't need to re-read from disk
    closs_sums = {lang: 0.0 for lang in LANGS}  # Mechanism B's JS consistency loss, tracked separately
                                                  # from the combined loss so you can actually see whether
                                                  # it's shrinking (regularizer doing something) or flat
                                                  # (not really constraining the model) -- previously
                                                  # computed every step but silently discarded, never logged.
    pbar = tqdm(range(1, iters + 1), desc=f"round={round_idx}", unit="it")
    for it in pbar:
        optimizer.zero_grad()
        loss_value, lang_metric, lang_closs = soft_lang_backward(
            model, tokenizer, train_datasets, raw_examples_by_lang, PER_LANG_BATCH_SIZE,
            letter_ids, device, rng, MAX_SEQ_LENGTH, tokenizer.pad_token_id, CONSISTENCY_LAMBDA,
        )
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()

        losses += loss_value
        steps += 1
        for lang in LANGS:
            metric_sums[lang] += lang_metric[lang]
            closs_sums[lang] += lang_closs[lang]
        pbar.set_postfix(loss=f"{loss_value:.4f}")

        if it % STEPS_PER_REPORT == 0 or it == iters:
            elapsed = time.time() - t0
            metric_str = " ".join(f"metric[{lang}]={metric_sums[lang]/steps:.3f}" for lang in LANGS)
            closs_str = " ".join(f"consistency[{lang}]={closs_sums[lang]/steps:.4f}" for lang in LANGS)
            tqdm.write(f"[{it}] train_loss(macro)={losses/steps:.4f} {metric_str} {closs_str} elapsed={elapsed:.0f}s")
            losses, steps = 0.0, 0
            metric_sums = {lang: 0.0 for lang in LANGS}
            closs_sums = {lang: 0.0 for lang in LANGS}

        if not is_refit and (it % STEPS_PER_EVAL == 0 or it == iters):
            val_loss, val_metric, lang_losses, lang_metrics = per_language_soft_validate(
                model, val_datasets, PER_LANG_BATCH_SIZE, VAL_BATCHES, letter_ids,
                tokenizer.pad_token_id, device, rng,
            )
            tqdm.write(f"[{it}] val_loss(macro)={val_loss:.4f} val_metric(macro)={val_metric:.4f}")
            stop_early = False
            if val_loss < best_val_loss:
                best_val_loss, best_iter, stale_evals = val_loss, it, 0
                model.save_pretrained(str(best_adapter_path))
                tqdm.write(f"[{it}] New best macro val loss; saved to {best_adapter_path}")
            else:
                stale_evals += 1
                tqdm.write(f"[{it}] No improvement ({stale_evals}/{PATIENCE})")
                if PATIENCE > 0 and stale_evals >= PATIENCE:
                    tqdm.write(f"[{it}] Early stopping.")
                    stop_early = True
            save_best_known(model, adapter_path, best_adapter_path)
            last_summary = {
                "cv_round": round_idx, "cv_folds": CV_FOLDS, "iter": it,
                "best_iter": best_iter, "best_val_loss": best_val_loss,
                "val_loss_macro": val_loss, "val_metric_macro": val_metric,
                "val_loss_by_lang": lang_losses, "val_metric_by_lang": lang_metrics,
                "consistency_lambda": CONSISTENCY_LAMBDA, "hard_labels": HARD_LABELS,
                "val_row_ids": {lang: [r.get("ID") for r in val_rows[lang]] for lang in LANGS},
                # False until the round actually completes below (ceiling reached or early
                # stopping fired) -- if this process gets killed/OOMs/restarts mid-round,
                # this on-disk copy stays "finished": False, so round_is_done correctly
                # retrains it from scratch instead of mistaking a half-finished round for done.
                "finished": False,
            }
            (out_dir / "summary.json").write_text(json.dumps(last_summary, indent=2))
            if stop_early:
                break

    if is_refit:
        model.save_pretrained(str(adapter_path))
        (out_dir / "summary.json").write_text(json.dumps({
            "final_refit": True, "refit_iters": iters,
            "consistency_lambda": CONSISTENCY_LAMBDA, "hard_labels": HARD_LABELS,
            "finished": True,
        }, indent=2))
        print(f"Final refit done. Adapter saved to {adapter_path}")
    else:
        save_best_known(model, adapter_path, best_adapter_path)
        print(f"CV round {round_idx}/{CV_FOLDS} done. best_iter={best_iter} best_val_loss={best_val_loss:.4f}")
        if last_summary is not None:
            last_summary["finished"] = True
            (out_dir / "summary.json").write_text(json.dumps(last_summary, indent=2))

    # This notebook runs multiple rounds in ONE process (unlike train_novel.py's
    # subprocess-per-round design, which gets a guaranteed-clean process every
    # round) -- so cleanup here has to be thorough, or memory accumulates round
    # over round until a later round OOMs even though round 0 fit fine.
    # optimizer holds direct references to every model parameter (and their
    # .grad buffers), so it must go before gc/empty_cache can actually reclaim
    # that memory, not just "del model" alone.
    del model, optimizer
    import gc
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.empty_cache()

    return out_dir / "summary.json"

## Run the CV sweep

In [ ]:
def round_is_done(round_idx, iters):
    """Checks the explicit "finished" flag run_round writes only on genuine
    completion (iteration ceiling reached or early stopping fired) -- NOT
    "iter >= best_iter", which is true after literally the first eval by
    construction (best_iter can never exceed the current iter) and would
    silently mark an interrupted/crashed round as done after 5 iterations."""
    summary_path = round_out_dir(round_idx) / "summary.json"
    if not summary_path.exists():
        return False
    try:
        summary = json.loads(summary_path.read_text())
    except (json.JSONDecodeError, OSError):
        return False
    return summary.get("finished", False)


for r in range(CV_FOLDS):
    if round_is_done(r, ITERS):
        print(f"round {r}/{CV_FOLDS} already done -- skipping (resume).")
        continue
    run_round(r, ITERS)

## Aggregate CV results -> suggested refit iteration count

In [ ]:
def aggregate_cv_results():
    val_losses, val_metrics, best_iters = [], [], []
    for r in range(CV_FOLDS):
        summary_path = round_out_dir(r) / "summary.json"
        if not summary_path.exists():
            print(f"round {r} has no summary.json yet -- run the CV sweep cell first.")
            return None
        summary = json.loads(summary_path.read_text())
        val_losses.append(summary["val_loss_macro"])
        val_metrics.append(summary["val_metric_macro"])
        best_iters.append(summary["best_iter"])

    median_iter = int(statistics.median(best_iters))
    refit_iters = ((median_iter + STEPS_PER_EVAL - 1) // STEPS_PER_EVAL) * STEPS_PER_EVAL

    agg = {
        "cv_folds": CV_FOLDS,
        "val_loss_macro_mean": statistics.mean(val_losses),
        "val_loss_macro_std": statistics.pstdev(val_losses) if len(val_losses) > 1 else 0.0,
        "val_metric_macro_mean": statistics.mean(val_metrics),
        "val_metric_macro_std": statistics.pstdev(val_metrics) if len(val_metrics) > 1 else 0.0,
        "best_iters": best_iters,
        "median_best_iter": median_iter,
        "suggested_refit_iters": refit_iters,
    }
    (OUT_BASE_DIR / "aggregate.json").write_text(json.dumps(agg, indent=2))
    print(f"{CV_FOLDS}-fold CV complete.")
    print(f"  val_loss_macro:   {agg['val_loss_macro_mean']:.4f} +/- {agg['val_loss_macro_std']:.4f}")
    print(f"  val_metric_macro: {agg['val_metric_macro_mean']:.4f} +/- {agg['val_metric_macro_std']:.4f}")
    print(f"  best_iters per fold: {best_iters} -> median {median_iter}, suggested refit_iters={refit_iters}")
    print(f"\n  Set REFIT_ITERS = {refit_iters} in the CONFIG cell above, then run the 'Final refit' cell below.")
    return agg

aggregate_cv_results()

## Final refit -- set `REFIT_ITERS` in the CONFIG cell above (use the value the aggregate cell suggested), then run this cell

In [ ]:
if REFIT_ITERS is None:
    print("Set REFIT_ITERS in the CONFIG cell above (use the value the aggregate cell suggested), "
          "then re-run this cell.")
else:
    run_round(None, REFIT_ITERS)